Quality check class to inspect schema, missing - and duplicated values

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

class SparkQCheck():
    """
    Class for performing quality checks on a Spark DataFrame.
    
    """

    def __init__(self,spark_df: DataFrame) -> None:
        self.spark_df=spark_df

    def __repr__(self) -> str:
        return f"Spark Dataframe {self.spark_df} loaded"

    def get_schema(self):
        """
        Returns the schema of the Spark DataFrame.

        """
        return self.spark_df.printSchema()

    def null_check_report(self) -> DataFrame:
        """
        Returns a DataFrame containing the count and percentage of null values for each column in the Spark DataFrame.
 
        """
        total = self.spark_df.count()
        exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in self.spark_df.columns]
        null_counts = self.spark_df.agg(*exprs).collect()[0].asDict()
        return self.spark_df.sparkSession.createDataFrame(
            [(c, n, round(n / total * 100, 2)) for c, n in null_counts.items()],
            ["column", "null_count", "null_pct"]
        )

    def duplicate_check_report(self) -> DataFrame:
        """
        Returns a DataFrame containing the count of duplicate rows in the Spark DataFrame.

        """
        return self.spark_df.groupBy(self.spark_df.columns).count().filter(F.col("count") > 1)
